In [ ]:
import warnings
warnings.filterwarnings('ignore')
import gc
import os
import math
import random
import pickle
import numpy as np
import pandas
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
random_seed_list = list(range(20))   # 20 random seeds (manuscript ensemble)
folder = '.'                          # run from the Code_availability/ root (contains Data/ and Results/)
results_folder_name = "Results_dco2_5ppm"
figure_path = os.path.join(folder, 'Results', results_folder_name, 'Figures')

Pre-Processing

In [ ]:
crop = 'Soy'
start_year_base = 1979
end_year = 2023 # Corrected end year to make the filter valid
year_datapoint_threshold = int((end_year - start_year_base) * 0.8)

In [ ]:
crop_df = pandas.read_csv(os.path.join(folder, 'Data', 'Aggregated_yield_climate_co2', crop + '_CAMS_1979_2023_allcounties.csv'))
crop_df['FIPS'] = crop_df.FIPS.apply(lambda x: f"{int(x):05}")
crop_df = crop_df[(crop_df.Year <= end_year)&(crop_df.Year >= start_year_base)] # year filter
fips_count = crop_df.FIPS.value_counts()
fips = crop_df.FIPS.unique()
fips = [i for i in fips if fips_count[i] > year_datapoint_threshold] # per county has to have more than 12 years of data
crop_df = crop_df[crop_df.FIPS.isin(fips)].reset_index(drop=True)

In [ ]:

print(crop_df['FIPS'].nunique())

In [ ]:
cams = crop_df.copy()
cams['Yield_deviation'] = cams['Yield'].copy()
cams['Year_delta'] = cams['Year'].copy()
cams['Year_delta'] -= start_year_base
cams['Year'] = cams['Year'] - start_year_base +1
cams['Year_squared'] = cams['Year']**2

# Maize-derived background trend (median across 20 seeds, on the fixed reliable county set)
county_year_diff = pickle.load(
    open(f'{folder}/Results/corn_trend_per_seed/Corn_county_year_tech_median_trend.p', 'rb')
)

# Merge tech trend into cams
cams['FIPS'] = cams['FIPS'].astype(str).str.zfill(5)
cams['Year_actual'] = cams['Year'] + 1979 - 1  # Year=1 corresponds to 1979

county_year_diff['FIPS'] = county_year_diff['FIPS'].astype(str).str.zfill(5)

cams = cams.merge(
    county_year_diff[['FIPS', 'Year_actual', 'diff_smooth_median']].rename(columns={'diff_smooth_median': 'Tech_trend'}),
    on=['FIPS', 'Year_actual'],
    how='left'
)
cams = cams.dropna(subset=['Tech_trend']).reset_index(drop=True)

# Replace the time variable with the maize-derived agronomy proxy (Scenario 2)
cams['Year'] = cams['Tech_trend']

full_df = cams.copy()

# Restrict to the fixed reliable county set (705 counties)
reliable_counties = pd.read_csv(f'{folder}/Results/reliable_counties_per_seed/reliable_counties_aggregated.csv', dtype={'FIPS': str})
reliable_counties['FIPS'] = reliable_counties['FIPS'].str.zfill(5)
full_df = full_df[full_df['FIPS'].isin(reliable_counties['FIPS'])].copy()

In [ ]:
def create_features(df):
    df['Precipitation_Early'] = df[['ppt_May','ppt_Jun']].mean(axis=1)
    df['Precipitation_Mid'] = df[['ppt_Jul','ppt_Aug']].mean(axis=1)
    df['Precipitation_Late'] = df[['ppt_Sep','ppt_Oct']].mean(axis=1)

    df['Tmin_Early'] = df[['tmin_May','tmin_Jun']].mean(axis=1)
    df['Tmin_Mid'] = df[['tmin_Jul','tmin_Aug']].mean(axis=1)
    df['Tmin_Late'] = df[['tmin_Sep','tmin_Oct']].mean(axis=1)

    df['Tmax_Early'] = df[['tmax_May','tmax_Jun']].mean(axis=1)
    df['Tmax_Mid'] = df[['tmax_Jul','tmax_Aug']].mean(axis=1)
    df['Tmax_Late'] = df[['tmax_Sep','tmax_Oct']].mean(axis=1)

    df['Tmean_Early'] = df[['tmean_May','tmean_Jun']].mean(axis=1)
    df['Tmean_Mid'] = df[['tmean_Jul','tmean_Aug']].mean(axis=1)
    df['Tmean_Late'] = df[['tmean_Sep','tmean_Oct']].mean(axis=1)

    df['CO2_Early'] = df[['co2_May','co2_Jun']].mean(axis=1)
    df['CO2_Mid'] = df[['co2_Jul','co2_Aug']].mean(axis=1)
    df['CO2_Late'] = df[['co2_Sep','co2_Oct']].mean(axis=1)

    df['CO2'] = df[['CO2_Early','CO2_Mid','CO2_Late']].mean(axis=1)
    df['Precipitation'] = df[['Precipitation_Early','Precipitation_Mid','Precipitation_Late']].mean(axis=1)
    df['Tmean'] = df[['Tmean_Early','Tmean_Mid','Tmean_Late']].mean(axis=1)
    df['Tmax'] = df[['Tmax_Early','Tmax_Mid','Tmax_Late']].mean(axis=1)
    df['Tmin'] = df[['Tmin_Early','Tmin_Mid','Tmin_Late']].mean(axis=1)

    return df


def alter_input(df, input_cols):
    """
    Normalize all input features using relative anomaly:
        (value - mean) / mean

    - Mean is computed globally (training set assumed).
    - Year is treated the same as other variables for consistency.
    - county_feature_trends keeps (mean, mean) for backward compatibility.
    """

    df_norm = df.copy()

    # ---------------------------
    # 1. Global mean for each feature
    # ---------------------------
    feature_mean = {}

    for c in input_cols:
        mu = df[c].mean()
        feature_mean[c] = mu

        # Avoid division by zero
        if mu == 0:
            df_norm[c] = 0.0
        else:
            df_norm[c] = df[c] / mu

    # ---------------------------
    # 2. Build county_feature_trends (compatibility only)
    # ---------------------------
    # We store (mean, mean) so downstream code expecting
    # (center, scale) does not break.
    # Interpretation is no longer mean/std or min/max.

    county_feature_trends = [{}, {}]

    counties = df.FIPS.unique()
    for county in counties:
        county_feature_trends[0][county] = {}
        county_feature_trends[1][county] = {}

        for c in input_cols:
            county_feature_trends[0][county][c] = feature_mean[c]
            county_feature_trends[1][county][c] = feature_mean[c]

    return df_norm, county_feature_trends

def encode_and_bind(df, encode_col, spatial=1, temporal=1):
    '''
    Helper function which one hot encodes column col and appends the encoded columns to the original dataframe (not in place)
    '''
    dummies_spatial = pandas.get_dummies(df[[encode_col]],prefix='Spatial',dtype=int)
    dummies_temporal = pandas.get_dummies(df[[encode_col]],prefix='Temporal',dtype=int)
    dummies_temporal.values[dummies_temporal!=0] = df['Year']
    df_list = [df]
    if spatial:
        df_list.append(dummies_spatial)
    if temporal:
        df_list.append(dummies_temporal)
    res = pandas.concat(df_list,axis=1)
    return res

def train_test_split_county (df):
    '''
    Helper function to generate train and test sets for each model run.
    Each split involves passing 80% of each counties available data to the train set and 20% to the test set.
    This ensure each county has presence in the training set to help train the free parameters,
    and also ensures that records from across the temporal range are present in both training and test sets.

    Parameters:
        df - DataFrame - Input dataframe

    Returns:
        train - Training split of input dataframe
        test - Test split of input dataframe
    '''
    train = []
    test = []
    county_model_data = df.groupby(['FIPS'])
    county_model_data = dict(list(county_model_data))
    counties = df.FIPS.unique()
    for county in counties:
        county_df = county_model_data[(county,)].sample(frac=1.0).copy()

        county_df = county_df.sort_values('Year').copy()
        n = int(max(2,int(county_df.shape[0]*0.2)))

        early = county_df.iloc[:n,:].copy()
        late  = county_df.iloc[-n:,:].copy()
        mid = county_df.iloc[n:-n,:].copy()

        edge_n = int(max(1,int(early.shape[0]*0.2)))
        mid_n = math.ceil(0.2*county_df.shape[0]) - (2*edge_n)

        early = early.sample(frac=1.0).copy()
        late = late.sample(frac=1.0).copy()
        mid = mid.sample(frac=1.0).copy()

        train.append(early.iloc[edge_n:,:].copy())
        test.append(early.iloc[:edge_n,:].copy())

        train.append(late.iloc[edge_n:,:].copy())
        test.append(late.iloc[:edge_n,:].copy())

        train.append(mid.iloc[mid_n:,:].copy())
        test.append(mid.iloc[:mid_n,:].copy())

    train = pandas.concat(train)
    test = pandas.concat(test)
    return train, test

In [ ]:
temporal_features = ['Year']
bimonthly_climate_features = ['Precipitation_Early', 'Tmax_Early', 'Tmin_Early','Precipitation_Mid', 'Tmax_Mid', 'Tmin_Mid','Precipitation_Late', 'Tmax_Late','Tmin_Late']
bimonthly_co2_features = ['CO2_Early','CO2_Mid','CO2_Late']
output_cols = ['Yield']
output_deviation_cols = ['Yield_deviation']

In [ ]:
cams_bck = full_df.copy()
cams_bck['FIPS'] = cams_bck['FIPS'].apply(lambda x: f"{int(x):05}")

cams_bck = create_features(cams_bck)

cams_bck = cams_bck[["FIPS","Year_delta"]+temporal_features+bimonthly_climate_features+bimonthly_co2_features+output_cols+output_deviation_cols]
cams_ref = cams_bck.groupby("FIPS",as_index=True).mean()

cams_bck, ref_dic = alter_input(cams_bck,temporal_features+bimonthly_climate_features+bimonthly_co2_features+output_deviation_cols)
cams_bck = encode_and_bind(cams_bck,'FIPS',spatial=1,temporal=0)

df_cams = cams_bck.copy()

In [ ]:
cams_county_cols = [c for c in df_cams.columns if c.startswith('Spatial_')]
linear_baseline_cols = bimonthly_climate_features + cams_county_cols
linear_baseline_time_cols = temporal_features + bimonthly_climate_features + cams_county_cols
linear_baseline_co2_cols = bimonthly_climate_features + bimonthly_co2_features + cams_county_cols
linear_baseline_time_co2_cols = temporal_features + bimonthly_climate_features + bimonthly_co2_features + cams_county_cols

nn_baseline_cols = bimonthly_climate_features + cams_county_cols
nn_baseline_time_cols = temporal_features + bimonthly_climate_features + cams_county_cols
nn_baseline_co2_cols = bimonthly_climate_features + bimonthly_co2_features + cams_county_cols
nn_baseline_time_co2_cols = temporal_features + bimonthly_climate_features + bimonthly_co2_features + cams_county_cols

source_df = df_cams.copy()

nn_baseline = nn_baseline_cols.copy()
nn_time = nn_baseline_time_cols.copy()
nn_co2 = nn_baseline_co2_cols.copy()
nn_time_co2 = nn_baseline_time_co2_cols.copy()

linear_baseline = linear_baseline_cols.copy()
linear_time = linear_baseline_time_cols.copy()
linear_co2 = linear_baseline_co2_cols.copy()
linear_time_co2 = linear_baseline_time_co2_cols.copy()

source_ref = cams_ref.copy()

In [ ]:
out_dir = os.path.join(folder, 'Results', results_folder_name)
os.makedirs(out_dir, exist_ok=True)

pickle.dump(source_ref,     open(f'{out_dir}/{crop}_source_ref.p', 'wb'))
pickle.dump(nn_baseline,    open(f'{out_dir}/{crop}_baseline.p', 'wb'))
pickle.dump(nn_time,        open(f'{out_dir}/{crop}_basetime.p', 'wb'))
pickle.dump(nn_co2,         open(f'{out_dir}/{crop}_input.p', 'wb'))
pickle.dump(nn_time_co2,    open(f'{out_dir}/{crop}_time.p', 'wb'))
pickle.dump(linear_baseline,open(f'{out_dir}/{crop}_baseline_linear.p', 'wb'))
pickle.dump(linear_time,    open(f'{out_dir}/{crop}_basetime_linear.p', 'wb'))
pickle.dump(linear_co2,     open(f'{out_dir}/{crop}_input_linear.p', 'wb'))
pickle.dump(linear_time_co2,open(f'{out_dir}/{crop}_time_linear.p', 'wb'))

Modelling

In [ ]:
train_loss = []
train_r2 = []
test_loss = []
test_r2 = []

models_list = []
run_train_list = []
run_test_list = []

sensitivities = []
year_sensitivities = []
year2_sensitivities = []
sens_early_list = []
sens_mid_list = []
sens_late_list = []

In [ ]:
num_runs = len(random_seed_list)
deltaC = 10

In [ ]:
def linear_model(
    df,
    input,
    output,
    ref,
    sensitivity_list,
    year_sensitivity_list,
    beta_time_list,
    beta_co2_abs_list,
    beta_co2_rel_list,
    beta_year_abs_list,
    beta_year_rel_list,
    county_time_trend=0,
    deltaC=1.0,
    year_scale=1.0,
):
    seed = random_seed
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    df_copy = df.copy()
    if county_time_trend:
        for sc in cams_county_cols:
            df_copy[sc] = df_copy[sc] * df_copy["Year"]

    # ---------------------------------------------------------
    # 1. 计算每个县的历史平均产量
    # ---------------------------------------------------------
    tmp = df_copy[["FIPS", "Yield"]].copy()
    tmp["FIPS_str"] = tmp["FIPS"].astype(str).str.zfill(5)
    real_mean_yield = tmp.groupby("FIPS_str")["Yield"].mean().to_dict()

    crop_model_run_train, crop_model_run_test = train_test_split_county(df_copy)
    train_X = crop_model_run_train[input]
    train_Y = np.array(crop_model_run_train[output].values.flatten())
    test_X = crop_model_run_test[input]
    test_Y = np.array(crop_model_run_test[output].values.flatten())

    model = Ridge(alpha=0.2)
    model.fit(train_X, train_Y)
    coef = np.asarray(model.coef_).ravel()
    coef_map = dict(zip(input, coef))

    # ========================================================================
    # 打印Year和Year²系数
    # ========================================================================
    print("\n" + "="*70)
    print("时间趋势系数诊断")
    print("="*70)

    beta_year_raw = float(coef_map.get("Year", np.nan))
    beta_time_list.append(beta_year_raw)

    beta_year_abs = beta_year_raw / float(year_scale) if (year_scale and year_scale != 0) else np.nan
    beta_year_abs_list.append(beta_year_abs)

    print(f"Year系数 (raw):      {beta_year_raw:.6f}")
    print(f"Year系数 (absolute): {beta_year_abs:.6f}")

    if "Year_squared" in coef_map:
        beta_year2_raw = float(coef_map["Year_squared"])
        beta_year2_abs = beta_year2_raw / (float(year_scale)**2) if (year_scale and year_scale != 0) else np.nan

        print(f"Year²系数 (raw):     {beta_year2_raw:.9f}")
        print(f"Year²系数 (absolute): {beta_year2_abs:.9f}")

        if abs(beta_year2_abs) > 1e-6:
            if beta_year2_abs > 0:
                print("  → ⚠️ 技术进步在加速 (Year² > 0)")
            else:
                print("  → ⚠️ 技术进步在减速 (Year² < 0)")
        else:
            print("  → ✓ Year²系数接近0，技术进步接近线性")

        year_min = df_copy["Year"].min() if "Year" in df_copy.columns else 1
        year_max = df_copy["Year"].max() if "Year" in df_copy.columns else 41
        year_range = year_max - year_min

        year_mid = year_range / 2
        year2_contribution_mid = beta_year2_abs * (year_mid ** 2)
        year2_contribution_end = beta_year2_abs * (year_range ** 2)

        print(f"\nYear²对产量的贡献估算（Year范围: {year_min}-{year_max}）:")
        print(f"  中期（Year={year_mid:.0f}）: {year2_contribution_mid:.2f} bu/acre")
        print(f"  末期（Year={year_range:.0f}）: {year2_contribution_end:.2f} bu/acre")

        linear_contribution = beta_year_abs * year_range
        total_contribution = linear_contribution + year2_contribution_end
        year2_fraction = abs(year2_contribution_end / total_contribution) * 100 if total_contribution != 0 else 0

        print(f"  Year²占总技术进步的比例: {year2_fraction:.1f}%")
    else:
        print("Year²系数: 不在特征中")

    print("="*70 + "\n")

    # ------------------
    # Performance
    # ------------------
    def mape_drop_zero(y_true, y_pred, eps=1e-12):
        y_true = np.asarray(y_true).ravel()
        y_pred = np.asarray(y_pred).ravel()
        mask = np.abs(y_true) > eps
        if mask.sum() == 0:
            return np.nan
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    train_pred = model.predict(train_X).ravel()
    test_pred  = model.predict(test_X).ravel()

    train_mae  = mean_absolute_error(train_Y, train_pred)
    train_r2   = r2_score(train_Y, train_pred)
    train_mape = mape_drop_zero(train_Y, train_pred)

    test_mae  = mean_absolute_error(test_Y, test_pred)
    test_r2   = r2_score(test_Y, test_pred)
    test_mape = mape_drop_zero(test_Y, test_pred)

    all_Y    = np.concatenate([train_Y, test_Y])
    all_pred = np.concatenate([train_pred, test_pred])
    overall_r2   = r2_score(all_Y, all_pred)
    overall_mape = mape_drop_zero(all_Y, all_pred)

    print("zero-y in train/test:", np.sum(np.abs(train_Y) <= 1e-12), np.sum(np.abs(test_Y) <= 1e-12))
    print(f"train r2: {train_r2:.4f}  mape: {train_mape:.2f}%")
    print(f"test  r2: {test_r2:.4f}  mape: {test_mape:.2f}%")
    print(f"all   r2: {overall_r2:.4f} mape: {overall_mape:.2f}%")

    models_list.append(model)
    run_train_list.append(crop_model_run_train)
    run_test_list.append(crop_model_run_test)

    # ------------------
    # CO2 sensitivity
    # ------------------
    beta_co2_norm = {f: coef_map[f] for f in bimonthly_co2_features if f in coef_map}
    stage_sensitivity = {f: [] for f in bimonthly_co2_features}

    for split_name, split_df in [('test', crop_model_run_test), ('train', crop_model_run_train)]:

        abs_effects = []
        stage_abs_effects = {f: [] for f in bimonthly_co2_features}

        for _, row in split_df.iterrows():
            fips   = str(row["FIPS"]).zfill(5)
            me_abs = 0.0
            valid  = True
            stage_me = {}

            for f in bimonthly_co2_features:
                if f not in beta_co2_norm:
                    stage_me[f] = np.nan
                    continue
                mu = ref[0][fips][f]
                if (mu is None) or (mu == 0) or np.isnan(mu):
                    valid = False
                    stage_me[f] = np.nan
                else:
                    effect = beta_co2_norm[f] / mu
                    me_abs += effect
                    stage_me[f] = effect

            abs_effects.append(me_abs if valid else np.nan)
            for f in bimonthly_co2_features:
                stage_abs_effects[f].append(stage_me.get(f, np.nan))

        # Overall sensitivity
        S_run = []
        for abs_eff, fips_raw, actual_yield in zip(abs_effects, split_df["FIPS"], split_df["Yield"]):
            fips   = str(fips_raw).zfill(5)
            norm_y = actual_yield
            if (norm_y is None) or (norm_y == 0) or np.isnan(norm_y) or np.isnan(abs_eff):
                S_run.append(np.nan)
            else:
                S_run.append((abs_eff / norm_y) * 100.0)
        sensitivity_list.extend(S_run)

        # Per-stage sensitivity
        for f in bimonthly_co2_features:
            S_stage = []
            for abs_eff, fips_raw, actual_yield in zip(
                    stage_abs_effects[f], split_df["FIPS"], split_df["Yield"]):
                fips   = str(fips_raw).zfill(5)
                norm_y = actual_yield
                if (norm_y is None) or (norm_y == 0) or np.isnan(norm_y) or np.isnan(abs_eff):
                    S_stage.append(np.nan)
                else:
                    S_stage.append((abs_eff / norm_y) * 100.0)
            stage_sensitivity[f].extend(S_stage)

        if split_name == 'test':
            beta_co2_abs_list.append(np.nanmedian(abs_effects))
            beta_co2_rel_list.append(np.nanmedian(S_run))
            print(f"[test] Overall CO2 sensitivity: {np.nanmedian(S_run):.6f}%/ppm")
            for f in bimonthly_co2_features:
                S_stage_arr = np.array(stage_sensitivity[f])
                print(f"  [{f}] median={np.nanmedian(S_stage_arr):.6f}%/ppm")

    # ------------------
    # Year sensitivity (test + train, written to year_sensitivity_list)
    # ------------------
    all_year_rel = []
    for split_name, split_df in [('test', crop_model_run_test), ('train', crop_model_run_train)]:
        year_rel = []
        for fips_raw in split_df["FIPS"]:
            fips   = str(fips_raw).zfill(5)
            mean_y = real_mean_yield.get(fips, np.nan)
            if (mean_y is None) or (mean_y == 0) or np.isnan(mean_y) or np.isnan(beta_year_abs):
                year_rel.append(np.nan)
            else:
                year_rel.append(100.0 * beta_year_abs / mean_y)
        all_year_rel.extend(year_rel)

        if split_name == 'test':
            beta_year_rel_list.append(np.nanmean(year_rel))

    year_sensitivity_list.extend(all_year_rel)

    return stage_sensitivity

In [ ]:
print("Linear No Time No CO2")
sensitivity = []
year_sens = []
beta_time_list1 = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

beta_co2_abs_list1 = []
beta_co2_rel_list1 = []
beta_year_abs_list1 = []
beta_year_rel_list1 = []

run_df     = source_df.copy()
run_input  = linear_baseline.copy()
run_output = output_cols.copy()

yvals          = np.sort(run_df["Year"].dropna().unique())
delta_year_var = np.median(np.diff(yvals))
year_scale     = 1.0

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = linear_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sens,
        beta_time_list1,
        beta_co2_abs_list1, beta_co2_rel_list1,
        beta_year_abs_list1, beta_year_rel_list1,
        county_time_trend=0,
        deltaC=deltaC,
        year_scale=year_scale
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp = np.asarray(sensitivity, dtype=float)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    block = snp[(ni * i):(ni * (i + 1))]
    print(np.nanmedian(block))

print(np.nanmedian(snp), np.nanmax(snp), np.nanmin(snp))
sensitivities.append(sensitivity)

print("beta_CO2_abs (yield/ppm), median =", np.nanmedian(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     median =", np.nanmedian(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_CO2_abs (yield/ppm), STD    =", np.nanstd(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     STD    =", np.nanstd(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_Year_abs (yield/year), median =", np.nanmedian(np.asarray(beta_year_abs_list1, dtype=float)) * delta_year_var)
print("beta_Year_rel (%/year),     median =", np.nanmedian(np.asarray(beta_year_rel_list1, dtype=float)) * delta_year_var)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
print("Linear Time")
sensitivity = []
year_sens = []                    # ← 新增
beta_time_list1 = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

beta_co2_abs_list1 = []
beta_co2_rel_list1 = []
beta_year_abs_list1 = []
beta_year_rel_list1 = []

run_df     = source_df.copy()
run_input  = linear_time.copy()
run_output = output_cols.copy()

yvals          = np.sort(run_df["Year"].dropna().unique())
delta_year_var = np.median(np.diff(yvals))
year_scale     = 1.0

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = linear_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sens,               # ← 加在这里
        beta_time_list1,
        beta_co2_abs_list1, beta_co2_rel_list1,
        beta_year_abs_list1, beta_year_rel_list1,
        county_time_trend=0,
        deltaC=deltaC,
        year_scale=year_scale
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

In [ ]:
print("Linear CO2")
sensitivity = []
year_sens = []
beta_time_list1 = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

beta_co2_abs_list1 = []
beta_co2_rel_list1 = []
beta_year_abs_list1 = []
beta_year_rel_list1 = []

run_df     = source_df.copy()
run_input  = linear_co2.copy()
run_output = output_cols.copy()

yvals          = np.sort(run_df["Year"].dropna().unique())
delta_year_var = np.median(np.diff(yvals))
year_scale     = 1.0

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = linear_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sens, beta_time_list1,
        beta_co2_abs_list1, beta_co2_rel_list1,
        beta_year_abs_list1, beta_year_rel_list1,
        county_time_trend=0,
        deltaC=deltaC,
        year_scale=year_scale
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp = np.asarray(sensitivity, dtype=float)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    block = snp[(ni * i):(ni * (i + 1))]
    print(np.nanmedian(block))

print(np.nanmedian(snp), np.nanmax(snp), np.nanmin(snp))
sensitivities.append(sensitivity)

print("beta_CO2_abs (yield/ppm), median =", np.nanmedian(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     median =", np.nanmedian(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_CO2_abs (yield/ppm), STD    =", np.nanstd(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     STD    =", np.nanstd(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_Year_abs (yield/year), median =", np.nanmedian(np.asarray(beta_year_abs_list1, dtype=float)) * delta_year_var)
print("beta_Year_rel (%/year),     median =", np.nanmedian(np.asarray(beta_year_rel_list1, dtype=float)) * delta_year_var)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
print("Linear Time CO2")
sensitivity = []
year_sens = []
beta_time_list1 = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

beta_co2_abs_list1 = []
beta_co2_rel_list1 = []
beta_year_abs_list1 = []
beta_year_rel_list1 = []

run_df     = source_df.copy()
run_input  = linear_time_co2.copy()
run_output = output_cols.copy()

yvals          = np.sort(run_df["Year"].dropna().unique())
delta_year_var = np.median(np.diff(yvals))
year_scale     = 1.0

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = linear_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sens, beta_time_list1,
        beta_co2_abs_list1, beta_co2_rel_list1,
        beta_year_abs_list1, beta_year_rel_list1,
        county_time_trend=0,
        deltaC=deltaC,
        year_scale=year_scale
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp = np.asarray(sensitivity, dtype=float)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    block = snp[(ni * i):(ni * (i + 1))]
    print(np.nanmedian(block))

print(np.nanmedian(snp), np.nanmax(snp), np.nanmin(snp))
sensitivities.append(sensitivity)

print("beta_CO2_abs (yield/ppm), median =", np.nanmedian(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     median =", np.nanmedian(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_CO2_abs (yield/ppm), STD    =", np.nanstd(np.asarray(beta_co2_abs_list1, dtype=float)))
print("beta_CO2_rel (%/ppm),     STD    =", np.nanstd(np.asarray(beta_co2_rel_list1, dtype=float)))
print("beta_Year_abs (yield/year), median =", np.nanmedian(np.asarray(beta_year_abs_list1, dtype=float)) * delta_year_var)
print("beta_Year_rel (%/year),     median =", np.nanmedian(np.asarray(beta_year_rel_list1, dtype=float)) * delta_year_var)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
predictions = []
trains = []
scalers_list = []
co2_delta=deltaC
def nn_model(df, input, output, ref, sensitivity_list, year_sensitivity_list, county_time_trend=0):
    print("Number of input features:", len(input))

    # -----------------------
    # 0) seed
    # -----------------------
    seed = random_seed
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    df_copy = df.copy()

    # -----------------------
    # 1) optional county*time trend
    # -----------------------
    if county_time_trend:
        for sc in cams_county_cols:
            df_copy[sc] = df_copy[sc] * df_copy["Year"]

    # -----------------------
    # 2) real mean yield by county (raw), for % per ppm scaling
    # -----------------------
    tmp = df_copy[["FIPS", "Yield"]].copy()
    tmp["FIPS_str"] = tmp["FIPS"].astype(str).str.zfill(5)
    real_mean_yield = tmp.groupby("FIPS_str")["Yield"].mean().to_dict()

    # -----------------------
    # 3) county-level train/test split
    # -----------------------
    crop_model_run_train, crop_model_run_test = train_test_split_county(df_copy)

    # -----------------------
    # 4) train/val split INSIDE train only
    # -----------------------
    train_sub, val_sub = train_test_split_county(crop_model_run_train)

    train_X = train_sub[input]
    train_Y = train_sub[output]
    val_X   = val_sub[input]
    val_Y   = val_sub[output]

    test_X  = crop_model_run_test[input]
    test_Y  = crop_model_run_test[output]

    # -----------------------
    # 5) 数据标准化
    # -----------------------
    scaler_X = StandardScaler()
    scaler_Y = StandardScaler()

    train_X_scaled = scaler_X.fit_transform(train_X)
    val_X_scaled   = scaler_X.transform(val_X)
    test_X_scaled  = scaler_X.transform(test_X)

    train_Y_scaled = scaler_Y.fit_transform(train_Y.values.reshape(-1, 1)).ravel()
    val_Y_scaled   = scaler_Y.transform(val_Y.values.reshape(-1, 1)).ravel()
    test_Y_scaled  = scaler_Y.transform(test_Y.values.reshape(-1, 1)).ravel()

    # -----------------------
    # 6) model definition
    # -----------------------
    model = Sequential()

    if crop == "Soy":
        model.add(Dense(64, activation="relu", input_shape=(len(input),)))
        model.add(Dense(32, activation="relu"))
        model.add(Dense(16, activation="relu"))
        model.add(Dense(1))
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
            loss="mean_squared_error"
        )
    elif crop == "Corn":
        model.add(Dense(64, activation="relu", input_shape=(len(input),)))
        model.add(Dense(32, activation="relu"))
        model.add(Dense(16, activation="relu"))
        model.add(Dense(1))
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
            loss="mean_squared_error"
        )
    else:
        model.add(Dense(64, activation="relu", input_shape=(len(input),)))
        model.add(Dense(32, activation="relu"))
        model.add(Dense(16, activation="relu"))
        model.add(Dense(1))
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
            loss="mean_squared_error"
        )

    rlrop = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=20,
        min_lr=1e-7,
        verbose=1
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=100,
        restore_best_weights=True,
        verbose=1
    )

    # -----------------------
    # 7) sample weights
    # -----------------------
    def create_sample_weights(df_subset, emphasis_years={2012: 3.0, 2008: 2.0, 2016: 2.0}):
        weights = np.ones(len(df_subset))
        for year, weight in emphasis_years.items():
            year_mask = df_subset['Year'] == year
            weights[year_mask] = weight
            n_samples = year_mask.sum()
            if n_samples > 0:
                print(f"  Year {year}: {n_samples} samples with weight={weight}")
        return weights

    print("\n[Creating sample weights for emphasis years: 2012, 2008, 2016]")
    print("Train set:")
    train_weights = create_sample_weights(train_sub)
    print("Validation set:")
    val_weights = create_sample_weights(val_sub)
    print(f"Total train samples: {len(train_weights)}, weighted: {train_weights.sum():.0f}")
    print(f"Total val samples:   {len(val_weights)}, weighted: {val_weights.sum():.0f}\n")

    # -----------------------
    # 8) train
    # -----------------------
    history = model.fit(
        train_X_scaled, train_Y_scaled,
        sample_weight=train_weights,
        epochs=2000,
        batch_size=64,
        verbose=1,
        validation_data=(val_X_scaled, val_Y_scaled, val_weights),
        callbacks=[early_stop, rlrop]
    )

    loss     = history.history.get("loss", [])
    val_loss = history.history.get("val_loss", [])

    # -----------------------
    # 9) evaluate
    # -----------------------
    def _mape(y_true, y_pred, eps=1e-12):
        y_true = np.asarray(y_true).ravel()
        y_pred = np.asarray(y_pred).ravel()
        mask = np.abs(y_true) > eps
        if mask.sum() == 0:
            return np.nan
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    y_tr = np.array(train_Y.values.flatten())
    y_va = np.array(val_Y.values.flatten())
    y_te = np.array(test_Y.values.flatten())

    yhat_tr_scaled = model.predict(train_X_scaled, verbose=0).ravel()
    yhat_va_scaled = model.predict(val_X_scaled,   verbose=0).ravel()
    yhat_te_scaled = model.predict(test_X_scaled,  verbose=0).ravel()

    yhat_tr = scaler_Y.inverse_transform(yhat_tr_scaled.reshape(-1, 1)).ravel()
    yhat_va = scaler_Y.inverse_transform(yhat_va_scaled.reshape(-1, 1)).ravel()
    yhat_te = scaler_Y.inverse_transform(yhat_te_scaled.reshape(-1, 1)).ravel()

    r2_tr = r2_score(y_tr, yhat_tr)
    r2_va = r2_score(y_va, yhat_va)
    r2_te = r2_score(y_te, yhat_te)

    mae_tr = mean_absolute_error(y_tr, yhat_tr)
    mae_va = mean_absolute_error(y_va, yhat_va)
    mae_te = mean_absolute_error(y_te, yhat_te)

    mape_tr = _mape(y_tr, yhat_tr)
    mape_va = _mape(y_va, yhat_va)
    mape_te = _mape(y_te, yhat_te)

    y_all    = np.concatenate([y_tr, y_te])
    yhat_all = np.concatenate([yhat_tr, yhat_te])
    r2_all   = r2_score(y_all, yhat_all)
    mape_all = _mape(y_all, yhat_all)

    print(f"[NN] R2   train={r2_tr:.4f}  val={r2_va:.4f}  test={r2_te:.4f}  overall(tr+te)={r2_all:.4f}")
    print(f"[NN] MAE  train={mae_tr:.4f}  val={mae_va:.4f}  test={mae_te:.4f}")
    print(f"[NN] MAPE train={mape_tr:.2f}%  val={mape_va:.2f}%  test={mape_te:.2f}%  overall(tr+te)={mape_all:.2f}%")
    print(f"[NN] early-stopped at epoch ~ {len(loss)} (best weights restored)")

    # -----------------------
    # 10) append to global lists
    # -----------------------
    models_list.append(model)
    run_train_list.append(crop_model_run_train)
    run_test_list.append(crop_model_run_test)

    train_loss.append(mae_tr)
    train_r2.append(r2_tr)
    test_loss.append(mae_te)
    test_r2.append(r2_te)

    if 'predictions' in globals():
        predictions.append(yhat_te)

    if 'trains' in globals():
        train_pred_full = np.full(len(crop_model_run_train), np.nan)
        for i, idx in enumerate(train_sub.index):
            loc = crop_model_run_train.index.get_loc(idx)
            train_pred_full[loc] = yhat_tr[i]
        for i, idx in enumerate(val_sub.index):
            loc = crop_model_run_train.index.get_loc(idx)
            train_pred_full[loc] = yhat_va[i]
        trains.append(train_pred_full)

    if 'scalers_list' in globals():
        scalers_list.append((scaler_X, scaler_Y))

    # ========================================================================
    # 11) CO2 Sensitivity
    # ========================================================================
    print("\n[Computing CO2 sensitivity using numerical differentiation]")

    stage_sensitivity = {f: [] for f in bimonthly_co2_features}

    for split_name, split_df in [('test', crop_model_run_test), ('train', crop_model_run_train)]:
        print(f"\n[Computing CO2 sensitivity on {split_name} set]")

        # base prediction
        split_X_scaled   = scaler_X.transform(split_df[input])
        base_pred_scaled = model.predict(split_X_scaled, verbose=0).ravel()
        base_pred        = scaler_Y.inverse_transform(base_pred_scaled.reshape(-1, 1)).ravel()

        # ── 1. 联合 perturb：三个 stage 同时 +1 ppm → combined_abs ────────
        split_pert_all = split_df.copy()
        fips_arr  = split_pert_all["FIPS"].values
        feat_data = {f: split_pert_all[f].values.copy() for f in bimonthly_co2_features if f in input}

        for i, fips_raw in enumerate(fips_arr):
            fips_key = str(fips_raw).zfill(5)
            for f in bimonthly_co2_features:
                if f not in input:
                    continue
                mu = ref[0][fips_key][f]
                if mu == 0 or np.isnan(mu):
                    feat_data[f][i] = np.nan
                else:
                    feat_data[f][i] = split_pert_all[f].values[i] + co2_delta / mu

        for f, vals in feat_data.items():
            split_pert_all[f] = vals

        pert_all_X_scaled = scaler_X.transform(split_pert_all[input])
        pert_all_pred     = scaler_Y.inverse_transform(
            model.predict(pert_all_X_scaled, verbose=0).ravel().reshape(-1, 1)).ravel()
        combined_abs = pert_all_pred - base_pred

        # ── 2. 单独 perturb 每个 stage → stage_abs_effects（用于 per-stage 输出）──
        stage_abs_effects = {}
        for stage_feature in bimonthly_co2_features:
            if stage_feature not in input:
                stage_abs_effects[stage_feature] = np.full(len(split_df), np.nan)
                continue

            split_pert = split_df.copy()
            stage_vals = split_pert[stage_feature].values.copy()

            for i, fips_raw in enumerate(split_pert["FIPS"].values):
                fips_key = str(fips_raw).zfill(5)
                mu = ref[0][fips_key][stage_feature]
                if mu == 0 or np.isnan(mu):
                    stage_vals[i] = np.nan
                else:
                    stage_vals[i] = split_pert[stage_feature].values[i] + co2_delta / mu

            split_pert[stage_feature] = stage_vals
            pert_X_scaled = scaler_X.transform(split_pert[input])
            pert_pred     = scaler_Y.inverse_transform(
                model.predict(pert_X_scaled, verbose=0).ravel().reshape(-1, 1)).ravel()
            stage_abs_effects[stage_feature] = pert_pred - base_pred

        # ── 3. Overall sensitivity（用联合 perturb 的 combined_abs）────────
        S_run = []
        for abs_eff, fips_raw, actual_yield in zip(combined_abs, split_df["FIPS"], split_df["Yield"]):
            norm_y = actual_yield
            # OLD: norm_y = real_mean_yield.get(str(fips_raw).zfill(5), np.nan)
            if (norm_y is None) or (norm_y == 0) or np.isnan(norm_y):
                S_run.append(np.nan)
            else:
                S_run.append((abs_eff / norm_y) * 100.0 / co2_delta)
        sensitivity_list.extend(S_run)
        print(f"  [{split_name}] overall median={np.nanmedian(S_run):.6f}%")

        # ── 4. Per-stage sensitivity（用单独 perturb 的 stage_abs_effects）─
        for stage_feature in bimonthly_co2_features:
            S_stage = []
            for abs_eff, fips_raw, actual_yield in zip(
                    stage_abs_effects[stage_feature], split_df["FIPS"], split_df["Yield"]):
                norm_y = actual_yield
                # OLD: norm_y = real_mean_yield.get(str(fips_raw).zfill(5), np.nan)
                if (norm_y is None) or (norm_y == 0) or np.isnan(norm_y) or np.isnan(abs_eff):
                    S_stage.append(np.nan)
                else:
                    S_stage.append((abs_eff / norm_y) * 100.0 / co2_delta)
            stage_sensitivity[stage_feature].extend(S_stage)
            print(f"  [{split_name}] {stage_feature} median={np.nanmedian(S_stage):.6f}%")


    # ========================================================================
    # 12) Year Sensitivity
    # ========================================================================
    print("\n[Computing Year sensitivity using numerical differentiation]")
    # ========================================================================
    year_sensitivity_local = []

    year_feature  = next((f for f in input if 'Year' in f and 'squared' not in f.lower()), None)
    year2_feature = next((f for f in input if 'squared' in f.lower() or 'Year_squared' in f), None)

    if year_feature is None:
        print("  ⚠️ No Year feature found, skipping Year sensitivity")
    else:
        for split_name, split_df in [('test', crop_model_run_test), ('train', crop_model_run_train)]:
            print(f"\n[Computing Year sensitivity on {split_name} set]")

            split_X_scaled   = scaler_X.transform(split_df[input])
            base_pred_scaled = model.predict(split_X_scaled, verbose=0).ravel()
            base_pred        = scaler_Y.inverse_transform(base_pred_scaled.reshape(-1, 1)).ravel()

            deltaYear      = 1.0
            split_year_pert = split_df.copy()

            for idx, row in split_year_pert.iterrows():
                fips_key = str(row["FIPS"]).zfill(5)
                mu_year  = ref[0][fips_key].get(year_feature, 20.0)
                if mu_year == 0 or np.isnan(mu_year):
                    split_year_pert.loc[idx, year_feature] = np.nan
                else:
                    split_year_pert.loc[idx, year_feature] = row[year_feature] + deltaYear / mu_year
                    if year2_feature:
                        mu_year2     = ref[0][fips_key].get(year2_feature, mu_year ** 2)
                        year_original = row[year_feature] * mu_year
                        year_new      = year_original + deltaYear
                        if mu_year2 != 0 and not np.isnan(mu_year2):
                            split_year_pert.loc[idx, year2_feature] = (year_new ** 2) / mu_year2

            split_year_pert_X_scaled = scaler_X.transform(split_year_pert[input])
            year_pert_pred_scaled    = model.predict(split_year_pert_X_scaled, verbose=0).ravel()
            year_pert_pred           = scaler_Y.inverse_transform(year_pert_pred_scaled.reshape(-1, 1)).ravel()

            year_abs_effects = year_pert_pred - base_pred

            for year_abs_eff, fips_raw in zip(year_abs_effects, split_df["FIPS"]):
                fips_key = str(fips_raw).zfill(5)
                mean_y   = real_mean_yield.get(fips_key, np.nan)
                if (mean_y is None) or (mean_y == 0) or np.isnan(mean_y) or np.isnan(year_abs_eff):
                    year_sensitivity_local.append(np.nan)
                else:
                    year_sensitivity_local.append((year_abs_eff / mean_y) * 100.0 / deltaYear)

            print(f"  [{split_name}] median={np.nanmedian(year_sensitivity_local):.6f}%/year")

    year_sensitivity_list.extend(year_sensitivity_local)

    return stage_sensitivity


In [ ]:
print("NN No Time No CO2")
sensitivity = []
year_sensitivity_list = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

run_df    = source_df.copy()
run_input = nn_baseline.copy()
run_output = output_cols.copy()

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = nn_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sensitivity_list,
        county_time_trend=0
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp      = np.array(sensitivity)
year_snp = np.array(year_sensitivity_list)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    print(f"Median S: {np.nanmedian(snp[(ni*i):(ni*(i+1))])}")
    print(f"STD of S: {np.nanstd(snp[(ni*i):(ni*(i+1))])}")
    print(f"  Median Year Sensitivity: {np.nanmedian(year_snp[(ni*i):(ni*(i+1))]):.6f} %/year")
    print(f"  STD: {np.nanstd(year_snp[(ni*i):(ni*(i+1))]):.6f}")

print(np.nanmedian(snp), np.nanquantile(snp, 0.95), np.nanquantile(snp, 0.05))
sensitivities.append(sensitivity)
year_sensitivities.append(year_sensitivity_list)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

# ── Print summary ─────────────────────────────────────────────
print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
print("NN Time")
sensitivity = []
year_sensitivity_list = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

run_df    = source_df.copy()
run_input = nn_time.copy()
run_output = output_cols.copy()

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = nn_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sensitivity_list,
        county_time_trend=0
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp      = np.array(sensitivity)
year_snp = np.array(year_sensitivity_list)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    print(f"Median S: {np.nanmedian(snp[(ni*i):(ni*(i+1))])}")
    print(f"STD of S: {np.nanstd(snp[(ni*i):(ni*(i+1))])}")
    print(f"  Median Year Sensitivity: {np.nanmedian(year_snp[(ni*i):(ni*(i+1))]):.6f} %/year")
    print(f"  STD: {np.nanstd(year_snp[(ni*i):(ni*(i+1))]):.6f}")

print(np.nanmedian(snp), np.nanquantile(snp, 0.95), np.nanquantile(snp, 0.05))
sensitivities.append(sensitivity)
year_sensitivities.append(year_sensitivity_list)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

# ── Print summary ─────────────────────────────────────────────
print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
print("NN CO2")
sensitivity = []
year_sensitivity_list = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

run_df    = source_df.copy()
run_input = nn_co2.copy()
run_output = output_cols.copy()

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = nn_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sensitivity_list,
        county_time_trend=0
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp      = np.array(sensitivity)
year_snp = np.array(year_sensitivity_list)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    print(f"Median S: {np.nanmedian(snp[(ni*i):(ni*(i+1))])}")
    print(f"STD of S: {np.nanstd(snp[(ni*i):(ni*(i+1))])}")
    print(f"  Median Year Sensitivity: {np.nanmedian(year_snp[(ni*i):(ni*(i+1))]):.6f} %/year")
    print(f"  STD: {np.nanstd(year_snp[(ni*i):(ni*(i+1))]):.6f}")

print(np.nanmedian(snp), np.nanquantile(snp, 0.95), np.nanquantile(snp, 0.05))
sensitivities.append(sensitivity)
year_sensitivities.append(year_sensitivity_list)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

# ── Print summary ─────────────────────────────────────────────
print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
print("NN Time CO2")
sensitivity = []
year_sensitivity_list = []
stage_sens_early = []
stage_sens_mid   = []
stage_sens_late  = []

run_df    = source_df.copy()
run_input = nn_time_co2.copy()
run_output = output_cols.copy()

for i in range(num_runs):
    random_seed = random_seed_list[i]
    stage_sens = nn_model(
        run_df, run_input, run_output, ref_dic,
        sensitivity, year_sensitivity_list,
        county_time_trend=0
    )
    if stage_sens is not None:
        stage_sens_early.extend(stage_sens['CO2_Early'])
        stage_sens_mid.extend(stage_sens['CO2_Mid'])
        stage_sens_late.extend(stage_sens['CO2_Late'])

snp      = np.array(sensitivity)
year_snp = np.array(year_sensitivity_list)

ni = int(len(snp) / num_runs)
for i in range(num_runs):
    print(f"Median S: {np.nanmedian(snp[(ni*i):(ni*(i+1))])}")
    print(f"STD of S: {np.nanstd(snp[(ni*i):(ni*(i+1))])}")
    print(f"  Median Year Sensitivity: {np.nanmedian(year_snp[(ni*i):(ni*(i+1))]):.6f} %/year")
    print(f"  STD: {np.nanstd(year_snp[(ni*i):(ni*(i+1))]):.6f}")

print(np.nanmedian(snp), np.nanquantile(snp, 0.95), np.nanquantile(snp, 0.05))
sensitivities.append(sensitivity)
year_sensitivities.append(year_sensitivity_list)

sens_early_list.append(stage_sens_early)
sens_mid_list.append(stage_sens_mid)
sens_late_list.append(stage_sens_late)

# ── Print summary ─────────────────────────────────────────────
print("\n=== Stage Sensitivity Summary (median over all runs) ===")
print(f"CO2_Early (May-Jun): median={np.nanmedian(stage_sens_early):.6f} %/ppm  std={np.nanstd(stage_sens_early):.6f}")
print(f"CO2_Mid   (Jul-Aug): median={np.nanmedian(stage_sens_mid):.6f} %/ppm  std={np.nanstd(stage_sens_mid):.6f}")
print(f"CO2_Late  (Sep-Oct): median={np.nanmedian(stage_sens_late):.6f} %/ppm  std={np.nanstd(stage_sens_late):.6f}")
print(f"Overall combined:    median={np.nanmedian(sensitivity):.6f} %/ppm  std={np.nanstd(sensitivity):.6f}")

In [ ]:
pickle.dump(sensitivities,open(folder+'/Results/'+results_folder_name+'/'+crop+'_sensitivities.p','wb'))
pickle.dump(year_sensitivities,open(folder+'/Results/'+results_folder_name+'/'+crop+'_year_sensitivities.p','wb'))
pickle.dump(sens_early_list,open(folder+'/Results/'+results_folder_name+'/'+crop+'_early_sensitivities.p','wb'))
pickle.dump(sens_mid_list,open(folder+'/Results/'+results_folder_name+'/'+crop+'_mid_sensitivities.p','wb'))
pickle.dump(sens_late_list, open(folder+'/Results/'+results_folder_name+'/'+crop+'_late_sensitivities.p','wb'))
pickle.dump(train_loss,open(folder+'/Results/'+results_folder_name+'/'+crop+'_train_loss.p','wb'))
pickle.dump(train_r2,open(folder+'/Results/'+results_folder_name+'/'+crop+'_train_r2.p','wb'))
pickle.dump(test_loss,open(folder+'/Results/'+results_folder_name+'/'+crop+'_test_loss.p','wb'))
pickle.dump(test_r2,open(folder+'/Results/'+results_folder_name+'/'+crop+'_test_r2.p','wb'))

# pickle.dump(models_list,open(folder+'/Results/'+results_folder_name+'/'+crop+'_models_list.p','wb'))

# pickle.dump(run_test_list,open(folder+'/Results/'+results_folder_name+'/'+crop+'_run_test_list.p','wb'))
# pickle.dump(run_train_list,open(folder+'/Results/'+results_folder_name+'/'+crop+'_run_train_list.p','wb'))
# pickle.dump(scalers_list, open(folder+'/Results/'+results_folder_name+'/'+crop+'_scalers_list.p','wb'))
save_tasks = [
    (run_train_list, f'{crop}_run_train_list.p'),
    (run_test_list,  f'{crop}_run_test_list.p'),
    (scalers_list,   f'{crop}_scalers_list.p'),
    (models_list,    f'{crop}_models_list.p'),
    # 你可以继续把其它的变量也写在这里
]

print("Starting to save large files to Google Drive...")

for data, filename in save_tasks:
    file_path = os.path.join(folder, 'Results', results_folder_name, filename)

    # 使用 with 语句：确保文件在 dump 完后立即关闭，触发同步
    with open(file_path, 'wb') as f:
        pickle.dump(data, f)
    print(f"Successfully dumped: {filename}")

# 关键一步：强制将内存缓冲区的数据刷入磁盘（对 Drive 挂载尤其有用）
if hasattr(os, 'sync'):
    os.sync()

print("All files synced to disk. You can safely close the notebook now.")

# trend_info = {'beta_hat': beta_hat, 'year_center': year_center}
# pickle.dump(trend_info, open(folder+'/Results/'+results_folder_name+'/'+crop+'_trend_info.p','wb'))

In [ ]:
# 加载结果
models_list = pickle.load(open(folder+'/Results/'+results_folder_name+'/'+crop+'_models_list.p','rb'))
run_test_list = pickle.load(open(folder+'/Results/'+results_folder_name+'/'+crop+'_run_test_list.p','rb'))
run_train_list = pickle.load(open(folder+'/Results/'+results_folder_name+'/'+crop+'_run_train_list.p','rb'))

# ========== 新增：加载scalers和趋势信息 ==========
# scalers_list = pickle.load(open(folder+'/Results/'+results_folder_name+'/'+crop+'_scalers_list.p','rb'))
# trend_info = pickle.load(open(folder+'/Results/'+results_folder_name+'/'+crop+'_trend_info.p','rb'))
# beta_hat = trend_info['beta_hat']
# year_center = trend_info['year_center']
# ================================================

predictions = []
train_pred = []

ctp = 2
stp = ctp * 4

# ================================================================
# Linear模型（索引0-7）- 没有标准化，直接预测
# ================================================================

# H1 Baseline (0, 1)
predictions.append(models_list[0].predict(run_test_list[0][linear_baseline.copy()]))
predictions.append(models_list[1].predict(run_test_list[1][linear_baseline.copy()]))
train_pred.append(models_list[0].predict(run_train_list[0][linear_baseline.copy()]))
train_pred.append(models_list[1].predict(run_train_list[1][linear_baseline.copy()]))

# H1 Time (2, 3)
predictions.append(models_list[(ctp*1)+0].predict(run_test_list[(ctp*1)+0][linear_time.copy()]))
predictions.append(models_list[(ctp*1)+1].predict(run_test_list[(ctp*1)+1][linear_time.copy()]))
train_pred.append(models_list[(ctp*1)+0].predict(run_train_list[(ctp*1)+0][linear_time.copy()]))
train_pred.append(models_list[(ctp*1)+1].predict(run_train_list[(ctp*1)+1][linear_time.copy()]))

# H1 CO2 (4, 5)
predictions.append(models_list[(ctp*2)+0].predict(run_test_list[(ctp*2)+0][linear_co2.copy()]))
predictions.append(models_list[(ctp*2)+1].predict(run_test_list[(ctp*2)+1][linear_co2.copy()]))
train_pred.append(models_list[(ctp*2)+0].predict(run_train_list[(ctp*2)+0][linear_co2.copy()]))
train_pred.append(models_list[(ctp*2)+1].predict(run_train_list[(ctp*2)+1][linear_co2.copy()]))

# H2 Time+CO2 (6, 7)
predictions.append(models_list[(ctp*3)+0].predict(run_test_list[(ctp*3)+0][linear_time_co2.copy()]))
predictions.append(models_list[(ctp*3)+1].predict(run_test_list[(ctp*3)+1][linear_time_co2.copy()]))
train_pred.append(models_list[(ctp*3)+0].predict(run_train_list[(ctp*3)+0][linear_time_co2.copy()]))
train_pred.append(models_list[(ctp*3)+1].predict(run_train_list[(ctp*3)+1][linear_time_co2.copy()]))

# ================================================================
# NN模型（索引8-15）- 有标准化，需要特殊处理
# ================================================================

# 辅助函数：处理NN预测（带标准化）
def predict_nn_with_scaler(model_idx, feature_list, add_trend=False):
    """
    使用scaler进行NN预测

    参数:
    - model_idx: models_list中的索引
    - feature_list: 特征列表（如nn_baseline, nn_co2等）
    - add_trend: 是否需要加回趋势（仅对H2 time+CO2为True）
    """
    model = models_list[model_idx]
    scaler_X, scaler_Y = scalers_list[model_idx - stp]  # NN的scaler从索引0开始

    # Test预测
    test_X = run_test_list[model_idx][feature_list]
    test_X_scaled = scaler_X.transform(test_X)
    test_pred_scaled = model.predict(test_X_scaled, verbose=0)
    test_pred = scaler_Y.inverse_transform(test_pred_scaled)

    # 如果需要加回趋势
    if add_trend:
        trend_test = beta_hat * (run_test_list[model_idx]["Year"].values - year_center)
        test_pred = test_pred.ravel() + trend_test
        test_pred = test_pred.reshape(-1, 1)

    # Train预测
    train_X = run_train_list[model_idx][feature_list]
    train_X_scaled = scaler_X.transform(train_X)
    train_pred_scaled = model.predict(train_X_scaled, verbose=0)
    train_pred_single = scaler_Y.inverse_transform(train_pred_scaled)

    # 如果需要加回趋势
    if add_trend:
        trend_train = beta_hat * (run_train_list[model_idx]["Year"].values - year_center)
        train_pred_single = train_pred_single.ravel() + trend_train
        train_pred_single = train_pred_single.reshape(-1, 1)

    return test_pred, train_pred_single

# H1 NN Baseline (8, 9)
test_p, train_p = predict_nn_with_scaler(stp+0, nn_baseline.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

test_p, train_p = predict_nn_with_scaler(stp+1, nn_baseline.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

# H1 NN Time (10, 11)
test_p, train_p = predict_nn_with_scaler(stp+(ctp*1)+0, nn_time.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

test_p, train_p = predict_nn_with_scaler(stp+(ctp*1)+1, nn_time.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

# H1 NN CO2 (12, 13)
test_p, train_p = predict_nn_with_scaler(stp+(ctp*2)+0, nn_co2.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

test_p, train_p = predict_nn_with_scaler(stp+(ctp*2)+1, nn_co2.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

# H2 NN Time+CO2 (14, 15) - 需要加回趋势！
test_p, train_p = predict_nn_with_scaler(stp+(ctp*3)+0, nn_time_co2.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

test_p, train_p = predict_nn_with_scaler(stp+(ctp*3)+1, nn_time_co2.copy(), add_trend=False)
predictions.append(test_p)
train_pred.append(train_p)

print(f"Total predictions: {len(predictions)}")
print(f"Total train_pred: {len(train_pred)}")

# 保存最终结果
pickle.dump(predictions, open(folder+'/Results/'+results_folder_name+'/'+crop+'_predictions.p','wb'))
pickle.dump(train_pred, open(folder+'/Results/'+results_folder_name+'/'+crop+'_train.p','wb'))

print("✓ 预测值已保存（包含inverse transform和趋势还原）")